# 📈 Chapter 4: Regression and Prediction
**Reference:** *Practical Statistics for Data Scientists (2nd Edition)* by Peter Bruce, Andrew Bruce, & Peter Gedeck

---

## 4.1 Introduction: Explanation vs. Prediction
Regression is arguably the most important statistical method in data science. However, it is crucial to understand that classical statisticians and modern data scientists use regression for fundamentally different reasons.

- **The Traditional Statistician (Explanation):** Uses regression to understand the exact nature of the relationship between variables. They care deeply about $p$-values, confidence intervals, and ensuring all mathematical assumptions (like normally distributed residuals) are perfectly met. Their goal is to say: *"A 1-unit increase in X causes a specific change in Y."*
- **The Data Scientist (Prediction):** Uses regression to predict future outcomes. They care primarily about out-of-sample accuracy, RMSE (Root Mean Squared Error), and cross-validation. If the model predicts accurately on unseen data, they are often willing to forgive minor violations of classical statistical assumptions.

In this chapter, we will bridge these two worlds. We will build regression models, evaluate their statistical summaries, interpret their coefficients, and diagnose their flaws.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# statsmodels is the best library for classical statistical regression in Python
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error, r2_score

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
np.random.seed(42)

print("Libraries successfully imported for Chapter 4.")

## 4.2 Simple Linear Regression
Simple linear regression models the relationship between a single predictor variable $X$ and a response variable $Y$ as a straight line:
$$ Y = b_0 + b_1X + e $$

- **$b_0$ (Intercept):** The predicted value of $Y$ when $X = 0$.
- **$b_1$ (Slope/Coefficient):** The expected change in $Y$ for every 1-unit increase in $X$.
- **$e$ (Error/Residual):** The difference between the predicted value (the line) and the actual data point.

The model finds the "best" line using **Ordinary Least Squares (OLS)**, which minimizes the sum of the squared residuals ($e^2$).

### Creating a Synthetic Housing Dataset
The book relies heavily on a King County (Seattle) housing dataset. To make this notebook reproducible, we will generate synthetic data that closely mimics the behavior of real estate data.

In [ ]:
# Generate synthetic King County-style housing data
n_houses = 500

# Square footage (Predictor)
sqft_living = np.random.normal(2000, 600, n_houses)

# Base price calculation (Slope = 250 per sqft, Intercept = 50,000)
# Adding a non-linear noise component to simulate real-world variance
true_error = np.random.normal(0, 80000, n_houses) + (sqft_living * np.random.normal(0, 30, n_houses))
sale_price = 50000 + (250 * sqft_living) + true_error

# Create DataFrame
house_df = pd.DataFrame({'SqFt': sqft_living, 'SalePrice': sale_price})

# 1. Fit Simple Linear Regression using statsmodels
# The '~' symbol means "SalePrice is predicted by SqFt"
simple_model = smf.ols('SalePrice ~ SqFt', data=house_df).fit()

print(simple_model.summary().tables[1])

# 2. Visualizing the OLS Regression Line
plt.figure(figsize=(8, 5))
sns.regplot(x='SqFt', y='SalePrice', data=house_df, 
            scatter_kws={'alpha':0.4, 'color':'gray'},
            line_kws={'color':'red', 'linewidth':3})
plt.title('Simple Linear Regression: House Price vs. Square Footage')
plt.xlabel('Square Feet')
plt.ylabel('Sale Price ($)')
plt.show()

print("Interpretation:")
print(f"For every 1 additional square foot, the model expects the house price to increase by ${simple_model.params['SqFt']:.2f}.")

## 4.3 Multiple Linear Regression
When we have multiple predictors, the equation extends:
$$ Y = b_0 + b_1X_1 + b_2X_2 + \\dots + b_nX_n + e $$

### Assessing the Model (Data Science vs. Statistics)
1. **RMSE (Root Mean Squared Error):** The primary metric for data scientists. It represents the average distance (error) between the model's predictions and the actual data. It is expressed in the same units as the target variable (e.g., Dollars).
2. **$R^2$ (R-Squared):** The proportion of variance in the target variable explained by the model. It ranges from 0 to 1. A classical statistician loves $R^2$, but data scientists are wary of it because $R^2$ artificially inflates as you add more variables, even if they are useless.
3. **Adjusted $R^2$:** Penalizes the model for adding useless predictors.
4. **t-statistic and p-value:** Used to determine if a specific predictor is statistically significant ($p < 0.05$). Data scientists usually prefer Cross-Validation over p-values to determine variable usefulness.

In [ ]:
# Adding more variables to our dataset
house_df['Bedrooms'] = np.round(house_df['SqFt'] / 800 + np.random.normal(1, 0.5, n_houses))
house_df['Age_Years'] = np.random.uniform(1, 100, n_houses)

# Modifying Sale Price to be affected by these new variables
house_df['SalePrice'] = house_df['SalePrice'] - (500 * house_df['Age_Years']) + (10000 * house_df['Bedrooms'])

# Fit Multiple Linear Regression
multi_model = smf.ols('SalePrice ~ SqFt + Bedrooms + Age_Years', data=house_df).fit()

print(multi_model.summary())

# Calculate RMSE explicitly
predictions = multi_model.predict(house_df)
rmse = np.sqrt(mean_squared_error(house_df['SalePrice'], predictions))
print(f"\nRoot Mean Squared Error (RMSE): ${rmse:,.2f}")
print("This means our model's predictions are, on average, off by about this dollar amount.")

## 4.4 Factor (Categorical) Variables in Regression
Regression equations require numbers. If we have a categorical variable (e.g., `Property_Type` with values like Single Family, Condo, Townhouse), we must convert it into numbers.

### Dummy Variables and Reference Coding
The standard approach is one-hot encoding (creating a binary 0/1 column for each category). However, in regression, including all dummy variables *plus* the intercept creates perfect multicollinearity (The Dummy Variable Trap).

To fix this, statistical software uses **Reference Coding** (drop-one encoding). It arbitrarily drops one category to serve as the baseline (the intercept). The coefficients of the remaining dummy variables represent the *difference* relative to that baseline.

In [ ]:
# Create a categorical variable
property_types = ['Single_Family', 'Townhouse', 'Condo']
house_df['PropType'] = np.random.choice(property_types, n_houses)

# Adjust prices based on property type
price_adjustments = {'Single_Family': 20000, 'Townhouse': 0, 'Condo': -30000}
house_df['SalePrice'] += house_df['PropType'].map(price_adjustments)

# Fit model with the categorical variable
# statsmodels automatically handles dummy encoding if the variable is a string/category
factor_model = smf.ols('SalePrice ~ SqFt + PropType', data=house_df).fit()

print(factor_model.summary().tables[1])

print("\nExplanation of Reference Coding:")
print("Notice that 'Condo' is missing from the table! Condo was chosen as the baseline (embedded in the Intercept).")
print(f"The coefficient for Single_Family means: 'Compared to a Condo, a Single Family home is worth ${factor_model.params['PropType[T.Single_Family]']:.2f} more, holding SqFt constant.'")

## 4.5 Interpreting the Regression Equation
Data scientists often treat models as "black boxes," but regression is highly interpretable. However, this interpretability comes with severe traps.

### 1. Correlated Predictors (Multicollinearity)
If two predictor variables are highly correlated with each other (e.g., `Bedrooms` and `SqFt`), the model struggles to figure out which variable is actually responsible for the change in `SalePrice`. The coefficients become wildly unstable. This is why you should check the correlation matrix before modeling.

### 2. Confounding Variables
If you leave out an important variable, your model can spit out nonsense. For example, if you regress `SalePrice` on `ZipCode` without including `SqFt`, a Zip Code with tiny apartments might look extremely cheap, not because the neighborhood is bad, but simply because the houses are smaller.

### 3. Interactions and Main Effects
Sometimes the effect of one variable depends entirely on the value of another. For example, an extra bedroom adds significant value to a large house, but might actually *decrease* the value of a tiny 800 SqFt house (because it makes the rooms too cramped). We capture this using an **Interaction Term** (`SqFt * Bedrooms`).

In [ ]:
# Fitting a model with an interaction term
# The syntax 'SqFt * Bedrooms' tells statsmodels to include both main effects AND their interaction
interaction_model = smf.ols('SalePrice ~ SqFt * Bedrooms', data=house_df).fit()

print(interaction_model.summary().tables[1])

print("\nThe 'SqFt:Bedrooms' row shows how the value of a square foot changes depending on how many bedrooms are squeezed into it.")

## 4.6 Regression Diagnostics
In classical statistics, a model is only valid if the residuals (errors) follow strict rules. Data scientists check these to ensure the model isn't completely blind to a pattern.

1. **Outliers:** Extreme values in the target variable ($Y$) that don't fit the model.
2. **High Leverage Points:** Extreme values in the predictor variables ($X$). They have a massive "pull" on the regression line.
3. **Cook’s Distance:** A metric combining outliers and leverage to identify **Influential Observations**—data points that drastically change the entire model if removed.
4. **Heteroskedasticity:** When the variance of the errors is not constant. (e.g., The model is very accurate for cheap houses, but wildly inaccurate for mansions). We detect this by plotting residuals against fitted values.

In [ ]:
# Calculating fitted values and residuals
fitted_values = simple_model.fittedvalues
residuals = simple_model.resid

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Residuals vs. Fitted Plot (Checking for Heteroskedasticity)
sns.scatterplot(x=fitted_values, y=residuals, ax=axes[0], alpha=0.5, color='purple')
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_title('Residuals vs. Fitted Values')
axes[0].set_xlabel('Predicted Sale Price')
axes[0].set_ylabel('Residual Error')

# 2. Histogram of Residuals (Checking for Normality)
sns.histplot(residuals, bins=30, kde=True, ax=axes[1], color='orange')
axes[1].set_title('Distribution of Residuals')
axes[1].set_xlabel('Residual Error')

plt.tight_layout()
plt.show()

print("Diagnostic Interpretation:")
print("If the Residuals vs Fitted plot looks like a funnel (wider on the right), we have Heteroskedasticity.")
print("This means our model's predictions become less reliable for very expensive houses.")

## 4.7 Polynomial and Spline Regression
What if the relationship isn't a straight line? 

- **Polynomial Regression:** We can add $X^2$ or $X^3$ to the model. While mathematically considered "linear regression" (because the coefficients $b_0, b_1$ are still linear), it draws a curved line through the data. Warning: Polynomials can swing wildly at the edges of the data (extrapolation).
- **Splines:** Instead of fitting one massive curve to the whole dataset, Splines cut the data into chunks (using 'knots') and fit a series of smooth, connected polynomial segments. This is a much more flexible and robust way to handle non-linearity.

In [ ]:
# Generating a clearly non-linear relationship
np.random.seed(99)
x_nonlin = np.linspace(1, 10, 100)
y_nonlin = np.sin(x_nonlin) * x_nonlin + np.random.normal(0, 1, 100)
nonlin_df = pd.DataFrame({'X': x_nonlin, 'Y': y_nonlin})

# Fit a standard linear model
lin_fit = smf.ols('Y ~ X', data=nonlin_df).fit()

# Fit a polynomial model (Degree 3)
# We use the I() function in statsmodels to compute transformations on the fly
poly_fit = smf.ols('Y ~ X + I(X**2) + I(X**3)', data=nonlin_df).fit()

plt.figure(figsize=(8, 5))
plt.scatter(nonlin_df['X'], nonlin_df['Y'], color='gray', label='Data')
plt.plot(nonlin_df['X'], lin_fit.fittedvalues, color='blue', linestyle='--', linewidth=2, label='Linear Fit (Poor)')
plt.plot(nonlin_df['X'], poly_fit.fittedvalues, color='red', linewidth=3, label='Polynomial Fit (Degree 3)')
plt.title('Capturing Non-Linearity with Polynomial Regression')
plt.legend()
plt.show()

### Conclusion of Chapter 4
Regression is the absolute bedrock of predictive modeling. 

**Key Takeaways for Data Scientists:**
1. **Prediction over Explanation:** While p-values and confidence intervals are mathematically elegant, always validate your models using out-of-sample metrics like RMSE and Cross-Validation.
2. **Beware the Dummy Trap:** Always use reference coding (dropping one category) when including categorical variables.
3. **Multicollinearity is a Silent Killer:** If your features are highly correlated, your coefficients will become completely unreliable, destroying the interpretability of your model.
4. **Residuals Tell the Truth:** The errors your model makes are not just garbage; plotting them reveals the structural flaws in your model (like heteroskedasticity or missed non-linear patterns).